## **MASt3R 3D Reconstruction for Gaussian Splat**




---

## **Pipeline Explanation**

This is a **complete 3D reconstruction pipeline** that transforms regular photos into interactive 3D scenes using Gaussian Splatting. It replaces traditional photogrammetry tools (COLMAP) with the modern **MASt3R model** for faster, more efficient processing.

### **Pipeline Steps:**

1. **Biplet-Square Normalization**
   - Converts images of any size into uniform square crops
   - Creates 2 views per image (Left/Right for landscape, Top/Bottom for portrait)
   - Preserves edge information that would be lost with single center crops

2. **DINO Pair Selection**
   - Uses DINOv2 (vision transformer) to find visually similar image pairs
   - Intelligently limits pairs to prevent memory overflow
   - Ensures good coverage across all images for robust reconstruction

3. **MASt3R Reconstruction**
   - Processes selected image pairs to estimate 3D geometry
   - Outputs: 3D point cloud + camera poses for each image
   - **Replaces the entire COLMAP pipeline** (no feature matching needed)

4. **COLMAP Format Conversion**
   - Converts MASt3R output to COLMAP-compatible binary format
   - Handles coordinate transformations (camera-to-world → world-to-camera)
   - Downsamples point cloud to manageable size (default: 1M points)

5. **Gaussian Splatting Training**
   - Trains a neural 3D representation from the reconstructed scene
   - Optimizes millions of 3D Gaussians to accurately reproduce the input views
   - Produces a real-time renderable 3D model

---

## **What is MASt3R?**

**MASt3R** (Matching and Stereo 3D Reconstruction) is a state-of-the-art AI model that solves two critical problems in 3D reconstruction simultaneously:

### **Traditional Approach (COLMAP):**
```
Images → Feature Detection (ALIKED) → Feature Matching (LightGlue) 
       → Geometric Verification → Sparse Reconstruction (COLMAP)
       → Dense Reconstruction
```
**Problem:** Multiple separate models, slow, requires careful parameter tuning

### **MASt3R Approach:**
```
Image Pairs → MASt3R Model → 3D Points + Camera Poses (done!)
```
**Advantage:** Single end-to-end model, faster, more robust

### **How MASt3R Works:**

1. **Input:** Takes pairs of overlapping images
2. **Processing:** 
   - Encodes images with Vision Transformer (ViT)
   - Predicts dense correspondences between images
   - Estimates depth for each pixel
   - Calculates relative camera poses
3. **Output:** 
   - Dense 3D point cloud
   - Camera positions and orientations
   - Confidence scores for each prediction

### **Key Benefits:**
- ✅ **No manual feature detection** needed
- ✅ **Works with challenging scenes** (textureless, reflective surfaces)
- ✅ **Faster than traditional pipelines**
- ✅ **Built-in global alignment** for multi-view consistency
- ✅ **Memory efficient** (processes pairs independently)

---

## **Why This Pipeline is Special:**

- **Memory-conscious:** Designed for limited hardware (16GB GPU)
- **Modular:** Each step can be run independently
- **Robust:** Combines classical vision (DINO) with modern learning (MASt3R)
- **End-to-end:** Raw photos → Interactive 3D model with one command
- **Production-ready:** Includes error handling, progress tracking, binary format conversion

This pipeline democratizes high-quality 3D reconstruction by making it accessible on consumer hardware while maintaining professional results.

---

# Setup

This technical script details a specialized computational pipeline designed to prepare data for Gaussian Splatting, a method used to generate high-quality 3D scenes from 2D images. The workflow stands out by integrating the MASt3R model to handle complex tasks like image matching and spatial positioning, effectively replacing traditional tools like COLMAP to streamline the reconstruction process. Beyond its core logic, the code prioritizes rigorous memory management and environment configuration, ensuring the system can handle intensive graphical processing within restricted hardware environments. Ultimately, this source serves as a comprehensive automation framework that installs necessary dependencies and optimizes the transition from raw visual data to a structured 3D environment.

In [1]:
# MASt3R-based Gaussian Splatting Pipeline
# Preserves: DINO pair selection + Biplet-Square Normalization
# Replaces: ALIKED/LightGlue/COLMAP with MASt3R

import os
import sys
import gc
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from pathlib import Path
import subprocess
from PIL import Image, ImageFilter
import struct

# Transformers for DINO
from transformers import AutoImageProcessor, AutoModel

# ============================================================================
# Configuration
# ============================================================================
class Config:
    # Feature extraction
    N_KEYPOINTS = 8192
    IMAGE_SIZE = 1024

    # Pair selection - CRITICAL for memory
    GLOBAL_TOPK = 20  # Reduced from 50 - each image pairs with top 20
    MIN_MATCHES = 10
    RATIO_THR = 1.2

    # Paths
    DINO_MODEL = "facebook/dinov2-base"
    
    # MASt3R - Reduced size for memory
    MAST3R_MODEL = "/kaggle/working/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
    MAST3R_IMAGE_SIZE = 224  # Small size to save memory

    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================================================================
# Memory Management Utilities
# ============================================================================

def clear_memory():
    """Aggressively clear GPU and CPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_memory_info():
    """Get current memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
    
    import psutil
    cpu_mem = psutil.virtual_memory().percent
    print(f"CPU Memory Usage: {cpu_mem:.1f}%")


# ============================================================================
# Environment Setup
# ============================================================================

def run_cmd(cmd, check=True, capture=False):
    """Run command with better error handling"""
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        capture_output=capture,
        text=True,
        check=False
    )
    if check and result.returncode != 0:
        print(f"❌ Command failed with code {result.returncode}")
        if capture:
            print(f"STDOUT: {result.stdout}")
            print(f"STDERR: {result.stderr}")
    return result


def setup_base_environment():
    """Setup base Python environment"""
    print("\n=== Setting up Base Environment ===")
    
    # NumPy fix for Python 3.12
    print("\n📦 Fixing NumPy...")
    run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
    run_cmd([sys.executable, "-m", "pip", "install", "numpy==1.26.4"])
    
    # PyTorch
    print("\n📦 Installing PyTorch...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision", "torchaudio"
    ])
    
    # Core utilities
    print("\n📦 Installing core utilities...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "opencv-python",
        "pillow",
        "imageio",
        "imageio-ffmpeg",
        "plyfile",
        "tqdm",
        "tensorboard",
        "scipy",  # for rotation conversions and image resizing
        "psutil"  # for memory monitoring
    ])
    
    # Transformers for DINO
    print("\n📦 Installing transformers...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "transformers==4.40.0"
    ])
    
    # pycolmap for COLMAP format
    print("\n📦 Installing pycolmap...")
    run_cmd([sys.executable, "-m", "pip", "install", "pycolmap"])
    
    print("✓ Base environment setup complete!")


def setup_mast3r():
    """Install and setup MASt3R"""
    print("\n=== Setting up MASt3R ===")
    
    os.chdir('/kaggle/working')
    
    # Remove existing installation
    if os.path.exists('mast3r'):
        print("Removing existing MASt3R installation...")
        os.system('rm -rf mast3r')
    
    # Clone repository
    print("Cloning MASt3R repository...")
    os.system('git clone --recursive https://github.com/naver/mast3r')
    os.chdir('/kaggle/working/mast3r')
    
    # Check dust3r directory
    print("Checking dust3r structure...")
    os.system('ls -la dust3r/')
    
    # Install dust3r
    print("Installing dust3r...")
    os.system('cd dust3r && python -m pip install -e .')
    
    # Install croco
    print("Installing croco...")
    os.system('cd dust3r/croco && python -m pip install -e .')
    
    # Install requirements
    print("Installing MASt3R requirements...")
    os.system('pip install -r requirements.txt')
    
    # Download model weights
    print("Downloading model weights...")
    os.system('mkdir -p checkpoints')
    os.system('wget -P checkpoints/ https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth')
    
    # Install additional dependencies
    print("Installing additional dependencies...")
    os.system('pip install trimesh matplotlib roma')
    
    # Add to path
    sys.path.insert(0, '/kaggle/working/mast3r')
    sys.path.insert(0, '/kaggle/working/mast3r/dust3r')
    
    # Verification
    print("\n🔍 Verifying MASt3R installation...")
    try:
        from mast3r.model import AsymmetricMASt3R
        print("  ✓ MASt3R import: OK")
    except Exception as e:
        print(f"  ❌ MASt3R import failed: {e}")
        raise
    
    print("✓ MASt3R setup complete!")

2026-01-26 18:05:24.432814: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769450724.615729      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769450724.670505      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769450725.134617      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769450725.134652      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769450725.134655      25 computation_placer.cc:177] computation placer alr

# Biplet

This code provides a systematic method for standardizing image datasets by transforming various photo dimensions into uniform, high-quality squares. The script functions by analyzing the orientation of an input file and then extracting two distinct, overlapping views—either left and right for landscape shots or top and bottom for portraits. By utilizing this dual-crop strategy, the program ensures that significant visual data from the edges of the original frame is preserved rather than lost to a single center cut. Ultimately, the process automates the normalization of visual data, resizing every output to a consistent resolution to prepare it for sophisticated computational tasks.

In [2]:
# ============================================================================
# Step 0: Biplet-Square Normalization (PRESERVED FROM ORIGINAL)
# ============================================================================

def normalize_image_sizes_biplet(input_dir, output_dir=None, size=1024):
    """
    Generates two square crops (Left & Right or Top & Bottom)
    from each image in a directory.
    """
    if output_dir is None:
        output_dir = 'output/images_biplet'

    os.makedirs(output_dir, exist_ok=True)

    print(f"Generating 2 cropped squares (Left/Right or Top/Bottom) for each image...")
    print()

    converted_count = 0
    size_stats = {}

    for img_file in sorted(os.listdir(input_dir)):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        input_path = os.path.join(input_dir, img_file)

        try:
            img = Image.open(input_path)
            original_size = img.size

            size_key = f"{original_size[0]}x{original_size[1]}"
            size_stats[size_key] = size_stats.get(size_key, 0) + 1

            # Generate 2 crops
            crops = generate_two_crops(img, size)

            base_name, ext = os.path.splitext(img_file)
            for mode, cropped_img in crops.items():
                output_path = os.path.join(output_dir, f"{base_name}_{mode}{ext}")
                cropped_img.save(output_path, quality=95)

            converted_count += 1
            print(f"  ✓ {img_file}: {original_size} → 2 square images generated")

        except Exception as e:
            print(f"  ✗ Error processing {img_file}: {e}")

    print(f"\nProcessing complete: {converted_count} source images processed")
    print(f"Original size distribution: {size_stats}")
    return converted_count


def generate_two_crops(img, size):
    """
    Crops the image into a square and returns 2 variations
    (Left/Right for landscape, Top/Bottom for portrait).
    """
    width, height = img.size
    crop_size = min(width, height)
    crops = {}

    if width > height:
        # Landscape → Left & Right
        positions = {
            'left': 0,
            'right': width - crop_size
        }
        for mode, x_offset in positions.items():
            box = (x_offset, 0, x_offset + crop_size, crop_size)
            crops[mode] = img.crop(box).resize(
                (size, size),
                Image.Resampling.LANCZOS
            )

    else:
        # Portrait or Square → Top & Bottom
        positions = {
            'top': 0,
            'bottom': height - crop_size
        }
        for mode, y_offset in positions.items():
            box = (0, y_offset, crop_size, y_offset + crop_size)
            crops[mode] = img.crop(box).resize(
                (size, size),
                Image.Resampling.LANCZOS
            )

    return crops

# Dino

This code outlines a specialized workflow for identifying similar image pairs within a large dataset to optimize computational efficiency. It begins by using a DINO-based neural network to extract high-level visual signatures, which allows the system to calculate the mathematical similarity between various files. To ensure the results are manageable, the script filters these connections by selecting the top-k most similar matches and applying a diversity-focused strategy that prevents any single image from dominating the selection. Ultimately, this process serves as an intelligent pre-selection phase, ensuring that subsequent analysis focuses on the most relevant pairs while maintaining broad coverage across the entire image collection.

In [3]:
# ============================================================================
# Step 1: DINO-based Pair Selection (PRESERVED FROM ORIGINAL)
# ============================================================================

def load_torch_image(fname, device):
    """Load image as torch tensor"""
    import torchvision.transforms as T

    img = Image.open(fname).convert('RGB')
    transform = T.Compose([
        T.ToTensor(),
    ])
    return transform(img).unsqueeze(0).to(device)

def extract_dino_global(image_paths, model_path, device):
    """Extract DINO global descriptors with memory management"""
    print("\n=== Extracting DINO Global Features ===")
    print("Initial memory state:")
    get_memory_info()

    processor = AutoImageProcessor.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path).eval().to(device)

    global_descs = []
    batch_size = 4  # Small batch to save memory
    
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        batch_imgs = []
        
        for img_path in batch_paths:
            img = load_torch_image(img_path, device)
            batch_imgs.append(img)
        
        batch_tensor = torch.cat(batch_imgs, dim=0)
        
        with torch.no_grad():
            inputs = processor(images=batch_tensor, return_tensors="pt", do_rescale=False).to(device)
            outputs = model(**inputs)
            desc = F.normalize(outputs.last_hidden_state[:, 1:].max(dim=1)[0], dim=1, p=2)
            global_descs.append(desc.cpu())
        
        # Clear batch memory
        del batch_tensor, inputs, outputs, desc
        clear_memory()

    global_descs = torch.cat(global_descs, dim=0)

    del model, processor
    clear_memory()
    
    print("After DINO extraction:")
    get_memory_info()

    return global_descs


def build_topk_pairs(global_feats, k, device):
    """Build top-k similar pairs from global features"""
    g = global_feats.to(device)
    sim = g @ g.T
    sim.fill_diagonal_(-1)

    N = sim.size(0)
    k = min(k, N - 1)

    topk_indices = torch.topk(sim, k, dim=1).indices.cpu()

    pairs = []
    for i in range(N):
        for j in topk_indices[i]:
            j = j.item()
            if i < j:
                pairs.append((i, j))

    # Remove duplicates
    pairs = list(set(pairs))
    
    return pairs


def select_diverse_pairs(pairs, max_pairs, num_images):
    """
    Select diverse pairs to ensure good image coverage
    Strategy: Select pairs that maximize image coverage
    """
    import random
    random.seed(42)
    
    if len(pairs) <= max_pairs:
        return pairs
    
    print(f"Selecting {max_pairs} diverse pairs from {len(pairs)} candidates...")
    
    # Count how many times each image appears in pairs
    image_counts = {i: 0 for i in range(num_images)}
    for i, j in pairs:
        image_counts[i] += 1
        image_counts[j] += 1
    
    # Sort pairs by: prefer pairs with less-connected images
    def pair_score(pair):
        i, j = pair
        # Lower score = images appear in fewer pairs = more diverse
        return image_counts[i] + image_counts[j]
    
    pairs_scored = [(pair, pair_score(pair)) for pair in pairs]
    pairs_scored.sort(key=lambda x: x[1])
    
    # Select pairs greedily to maximize coverage
    selected = []
    selected_images = set()
    
    # Phase 1: Select pairs that add new images (greedy coverage)
    for pair, score in pairs_scored:
        if len(selected) >= max_pairs:
            break
        i, j = pair
        # Prefer pairs that include new images
        if i not in selected_images or j not in selected_images:
            selected.append(pair)
            selected_images.add(i)
            selected_images.add(j)
    
    # Phase 2: Fill remaining slots with high-similarity pairs
    if len(selected) < max_pairs:
        remaining = [p for p, s in pairs_scored if p not in selected]
        random.shuffle(remaining)
        selected.extend(remaining[:max_pairs - len(selected)])
    
    print(f"Selected pairs cover {len(selected_images)} / {num_images} images ({100*len(selected_images)/num_images:.1f}%)")
    
    return selected


def get_image_pairs_dino(image_paths, max_pairs=None):
    """DINO-based pair selection with intelligent limiting"""
    device = Config.DEVICE

    # DINO global features
    global_feats = extract_dino_global(image_paths, Config.DINO_MODEL, device)
    pairs = build_topk_pairs(global_feats, Config.GLOBAL_TOPK, device)

    print(f"Initial pairs from DINO: {len(pairs)}")
    
    # Apply intelligent pair selection if limit specified
    if max_pairs and len(pairs) > max_pairs:
        pairs = select_diverse_pairs(pairs, max_pairs, len(image_paths))
    
    return pairs

# Mast3r


This code outlines a specialized workflow for 3D scene reconstruction that utilizes the MASt3R framework as a streamlined alternative to traditional photogrammetry pipelines. By integrating image loading, model inference, and global alignment, the script efficiently transforms a series of 2D photographs into a unified point cloud. The process is characterized by a strong focus on memory management, employing strategies such as image resizing and pair limitation to maintain performance on hardware with limited resources. Ultimately, the system automates the complex task of spatial optimization, calculating the most accurate geometric relationship between images to build a coherent digital environment.

In [4]:
# ============================================================================
# Step 2: MASt3R Reconstruction (REPLACES ALIKED/LIGHTGLUE/COLMAP)
# ============================================================================

def load_mast3r_model(device='cuda'):
    """Load MASt3R model"""
    from mast3r.model import AsymmetricMASt3R
    
    model = AsymmetricMASt3R.from_pretrained(Config.MAST3R_MODEL).to(device)
    model.eval()
    
    print(f"✓ MASt3R model loaded on {device}")
    return model

def load_images_for_mast3r(image_paths, size=224):
    """Load images using DUSt3R's format with reduced size"""
    print(f"\n=== Loading images for MASt3R (size={size}) ===")
    
    from dust3r.utils.image import load_images
    
    # Load images using DUSt3R's loader with reduced size
    images = load_images(image_paths, size=size, verbose=True)
    
    return images

def run_mast3r_pairs(model, image_paths, pairs, device='cuda', batch_size=1, max_pairs=None):
    """Run MASt3R on selected pairs with memory management"""
    print("\n=== Running MASt3R Reconstruction ===")
    print("Initial memory state:")
    get_memory_info()
    
    from dust3r.inference import inference
    from dust3r.cloud_opt import global_aligner, GlobalAlignerMode
    
    # Limit number of pairs if specified
    if max_pairs and len(pairs) > max_pairs:
        print(f"Limiting pairs from {len(pairs)} to {max_pairs}")
        # Select pairs more evenly distributed
        step = max(1, len(pairs) // max_pairs)
        pairs = pairs[::step][:max_pairs]
    
    print(f"Processing {len(pairs)} pairs...")
    
    # Load images in smaller size
    print(f"Loading {len(image_paths)} images at {Config.MAST3R_IMAGE_SIZE}x{Config.MAST3R_IMAGE_SIZE}...")
    images = load_images_for_mast3r(image_paths, size=Config.MAST3R_IMAGE_SIZE)
    
    print(f"Loaded {len(images)} images")
    print("After loading images:")
    get_memory_info()
    
    # Create all image pairs at once
    print(f"Creating {len(pairs)} image pairs...")
    mast3r_pairs = []
    for idx1, idx2 in tqdm(pairs, desc="Preparing pairs"):
        mast3r_pairs.append((images[idx1], images[idx2]))
    
    print(f"Running MASt3R inference on {len(mast3r_pairs)} pairs...")
    
    # Run inference (this returns the dict format we need)
    output = inference(mast3r_pairs, model, device, batch_size=batch_size, verbose=True)
    
    # Clear pairs from memory
    del mast3r_pairs
    clear_memory()
    
    print("✓ MASt3R inference complete")
    print("After inference:")
    get_memory_info()
    
    # Global alignment
    print("Running global alignment...")
    scene = global_aligner(
        output, 
        device=device, 
        mode=GlobalAlignerMode.PointCloudOptimizer
    )
    
    # Clear output after creating scene
    del output
    clear_memory()
    
    print("Computing global alignment...")
    loss = scene.compute_global_alignment(
        init="mst", 
        niter=150,  # Reduced from 300
        schedule='cosine', 
        lr=0.01
    )
    
    print(f"✓ Global alignment complete (final loss: {loss:.6f})")
    print("Final memory state:")
    get_memory_info()
    
    return scene, images

# Ps1(process1)

This source provides a Python script designed to bridge the gap between MASt3R scene reconstructions and COLMAP, a standard format for 3D computer vision data. The code systematically extracts 3D point clouds and camera trajectories, performing necessary mathematical adjustments such as inverting camera-to-world poses and scaling focal lengths to match original image dimensions. To ensure efficiency, the script includes a downsampling mechanism that limits the total number of spatial points, preventing memory overflow while maintaining scene geometry. Finally, it serializes this processed information into binary files, specifically cameras, images, and 3D points, allowing the reconstructed scene to be opened and utilized by other specialized 3D software.

In [5]:
#v26
def extract_colmap_data(scene, image_paths, max_points=1000000):
    """
    Extract COLMAP-compatible camera parameters and 3D points from MASt3R scene
    
    Args:
        scene: MASt3R scene object
        image_paths: List of image paths
        max_points: Maximum number of 3D points to extract (default: 1M)
    """
    print("\n=== Extracting COLMAP-compatible data ===")
    
    # Extract point cloud
    pts_all = scene.get_pts3d()
    print(f"pts_all type: {type(pts_all)}")
    
    if isinstance(pts_all, list):
        print(f"pts_all is a list with {len(pts_all)} elements")
        if len(pts_all) > 0:
            print(f"First element type: {type(pts_all[0])}")
            if hasattr(pts_all[0], 'shape'):
                print(f"First element shape: {pts_all[0].shape}")
        
        pts_all = torch.stack([p if isinstance(p, torch.Tensor) else torch.tensor(p) 
                              for p in pts_all])
        print(f"pts_all shape after conversion: {pts_all.shape}")
    
    if len(pts_all.shape) == 4:
        print(f"Found batched point cloud: {pts_all.shape}")
        B, H, W, _ = pts_all.shape
        pts3d = pts_all.reshape(-1, 3).detach().cpu().numpy()  
        
        # Extract colors
        colors = []
        for img_path in image_paths:
            img = Image.open(img_path).resize((W, H))
            colors.append(np.array(img))
        colors = np.stack(colors).reshape(-1, 3) / 255.0
    else:
        pts3d = pts_all.detach().cpu().numpy() if isinstance(pts_all, torch.Tensor) else pts_all
        colors = np.ones((len(pts3d), 3)) * 0.5
    
    print(f"✓ Extracted {len(pts3d)} 3D points from {len(image_paths)} images")
    
    # **DOWNSAMPLE POINTS TO REDUCE MEMORY USAGE**
    if len(pts3d) > max_points:
        print(f"\n⚠ Downsampling from {len(pts3d)} to {max_points} points to reduce memory usage...")
        
        # Remove invalid points first
        valid_mask = ~(np.isnan(pts3d).any(axis=1) | np.isinf(pts3d).any(axis=1))
        pts3d_valid = pts3d[valid_mask]
        colors_valid = colors[valid_mask]
        
        # Random sampling
        indices = np.random.choice(len(pts3d_valid), size=max_points, replace=False)
        pts3d = pts3d_valid[indices]
        colors = colors_valid[indices]
        
        print(f"✓ Downsampled to {len(pts3d)} points")
    
    # Extract camera parameters
    print("Extracting camera parameters...")
    
    # [IMPORTANT] MASt3R poses are in camera-to-world format.
    # COLMAP requires world-to-camera format, so we need the inverse matrix.
    poses_c2w = scene.get_im_poses().detach().cpu().numpy()
    print(f"Retrieved camera-to-world poses: shape {poses_c2w.shape}")
    
    # Convert camera-to-world to world-to-camera
    poses = []
    for i, pose_c2w in enumerate(poses_c2w):
        # Calculate the inverse of the 4x4 matrix
        pose_w2c = np.linalg.inv(pose_c2w)
        poses.append(pose_w2c)
    
    poses = np.array(poses)
    print(f"Converted to world-to-camera poses for COLMAP")
    
    # Get focal lengths and principal points
    focals = scene.get_focals().detach().cpu().numpy()
    pp = scene.get_principal_points().detach().cpu().numpy()
    print(f"Focals shape: {focals.shape}")
    print(f"Principal points shape: {pp.shape}")
    
    # MASt3R internal processing size (usually 224x224)
    mast3r_size = 224.0
    
    cameras = []
    for i, img_path in enumerate(image_paths):
        img = Image.open(img_path)
        W, H = img.size
        
        # Scale ratio relative to the original image size
        scale = W / mast3r_size
        
        # Focals are in [N, 1] format (fx=fy for isotropic cameras)
        if focals.shape[1] == 1:
            focal_mast3r = float(focals[i, 0])
            fx = fy = focal_mast3r * scale
        else:
            fx = float(focals[i, 0]) * scale
            fy = float(focals[i, 1]) * scale
        
        # Scale principal points as well
        cx = float(pp[i, 0]) * scale
        cy = float(pp[i, 1]) * scale
        
        camera = {
            'camera_id': i + 1,
            'model': 'PINHOLE',
            'width': W,
            'height': H,
            'params': [fx, fy, cx, cy]
        }
        cameras.append(camera)
        
        if i == 0:
            print(f"\nExample camera 0:")
            print(f"  Image size: {W}x{H}")
            print(f"  MASt3R focal: {focal_mast3r:.2f}, pp: ({pp[i,0]:.2f}, {pp[i,1]:.2f})")
            print(f"  Scaled fx={fx:.2f}, fy={fy:.2f}, cx={cx:.2f}, cy={cy:.2f}")
            print(f"  Pose (first row): {poses[i][0]}")
    
    print(f"\n✓ Extracted {len(cameras)} cameras and {len(poses)} poses")
    
    return pts3d, colors, cameras, poses


def save_colmap_reconstruction(pts3d, colors, cameras, poses, image_paths, output_dir):
    """Save reconstruction in COLMAP binary format by writing files directly"""
    print("\n=== Saving COLMAP reconstruction ===")
    
    sparse_dir = Path(output_dir) / 'sparse' / '0'
    sparse_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"  Writing COLMAP files directly to {sparse_dir}...")
    
    # Write cameras.bin
    write_cameras_binary(cameras, sparse_dir / 'cameras.bin')
    print(f"  ✓ Wrote {len(cameras)} cameras")
    
    # Write images.bin
    write_images_binary(image_paths, cameras, poses, sparse_dir / 'images.bin')
    print(f"  ✓ Wrote {len(image_paths)} images")
    
    # Write points3D.bin
    num_points = write_points3d_binary(pts3d, colors, sparse_dir / 'points3D.bin')
    print(f"  ✓ Wrote {num_points} 3D points")
    
    print(f"\n✓ COLMAP reconstruction saved to {sparse_dir}")
    print(f"  Cameras: {len(cameras)}")
    print(f"  Images: {len(image_paths)}")
    print(f"  Points: {num_points}")
    
    return sparse_dir


def write_cameras_binary(cameras, output_file):
    """Write cameras.bin in COLMAP binary format"""
    with open(output_file, 'wb') as f:
        # Write number of cameras
        f.write(struct.pack('Q', len(cameras)))
        
        for i, cam in enumerate(cameras):
            camera_id = cam.get('camera_id', i + 1)
            
            # Model ID: 1 = PINHOLE
            model_id = 1
            width = cam['width']
            height = cam['height']
            params = cam['params']  # [fx, fy, cx, cy]
            
            f.write(struct.pack('i', camera_id))
            f.write(struct.pack('i', model_id))
            f.write(struct.pack('Q', width))
            f.write(struct.pack('Q', height))
            
            # Write 4 parameters for PINHOLE model
            for param in params[:4]:
                f.write(struct.pack('d', param))


def write_images_binary(image_paths, cameras, poses, output_file):
    """Write images.bin in COLMAP binary format"""
    with open(output_file, 'wb') as f:
        # Write number of images
        f.write(struct.pack('Q', len(image_paths)))
        
        for i, (img_path, pose) in enumerate(zip(image_paths, poses)):
            image_id = i + 1
            camera_id = cameras[i].get('camera_id', i + 1)
            image_name = os.path.basename(img_path)
            
            # Extract rotation and translation
            R = pose[:3, :3]
            t = pose[:3, 3]
            
            # Convert rotation matrix to quaternion [w, x, y, z]
            qvec = rotmat2qvec(R)
            tvec = t
            
            # Write image data
            f.write(struct.pack('i', image_id))
            
            # Write quaternion (4 doubles)
            for q in qvec:
                f.write(struct.pack('d', float(q)))
            
            # Write translation vector (3 doubles)
            for tv in tvec:
                f.write(struct.pack('d', float(tv)))
            
            # Write camera ID
            f.write(struct.pack('i', camera_id))
            
            # Write image name (null-terminated string)
            f.write(image_name.encode('utf-8') + b'\x00')
            
            # Write number of 2D points (0 for now, as we don't have 2D-3D correspondences)
            f.write(struct.pack('Q', 0))


def write_points3d_binary(pts3d, colors, output_file):
    """Write points3D.bin in COLMAP binary format"""
    # Filter out invalid points
    valid_indices = []
    for i, pt in enumerate(pts3d):
        if not (np.isnan(pt).any() or np.isinf(pt).any()):
            valid_indices.append(i)
    
    with open(output_file, 'wb') as f:
        # Write number of points
        f.write(struct.pack('Q', len(valid_indices)))
        
        for idx, point_id in enumerate(valid_indices):
            pt = pts3d[point_id]
            color = colors[point_id]
            
            # Write point3D ID
            f.write(struct.pack('Q', point_id))
            
            # Write XYZ coordinates (3 doubles)
            for coord in pt:
                f.write(struct.pack('d', float(coord)))
            
            # Write RGB color (3 unsigned chars)
            col_int = (color * 255).astype(np.uint8)
            for c in col_int:
                f.write(struct.pack('B', int(c)))
            
            # Write error (1 double) - set to 0
            f.write(struct.pack('d', 0.0))
            
            # Write track length (number of images seeing this point)
            # Set to 0 as we don't have track information
            f.write(struct.pack('Q', 0))
            
            # Progress indicator
            if (idx + 1) % 1000000 == 0:
                print(f"    Wrote {idx + 1} / {len(valid_indices)} points...")
    
    return len(valid_indices)


def rotmat2qvec(R):
    """
    Convert rotation matrix to quaternion in COLMAP format [w, x, y, z]
    
    Args:
        R: 3x3 rotation matrix
        
    Returns:
        qvec: quaternion [w, x, y, z]
    """
    # Ensure R is a numpy array
    R = np.asarray(R, dtype=np.float64)
    
    # Calculate trace
    trace = np.trace(R)
    
    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s
    
    qvec = np.array([w, x, y, z], dtype=np.float64)
    
    # Normalize
    qvec = qvec / np.linalg.norm(qvec)
    
    return qvec

# Gs(gaussian splat)

This script provides a technical framework for implementing Gaussian Splatting, a cutting-edge method used to generate high-quality 3D reconstructions from 2D images. The process begins by configuring the environment, which involves cloning necessary repositories and installing specialized submodules for rasterization and data processing. Once the infrastructure is ready, the code executes a training phase where it optimizes a point cloud model based on visual data. To ensure efficiency in resource-constrained environments, the script employs performance-tuning parameters such as reduced image resolution and controlled densification intervals. Ultimately, this workflow serves as an automated pipeline for transforming raw spatial data into a fully rendered 3D scene.

In [6]:
# ============================================================================
# Step 3: Gaussian Splatting Training
# ============================================================================

def setup_gaussian_splatting():
    """Setup Gaussian Splatting"""
    print("\n=== Setting up Gaussian Splatting ===")
    
    os.chdir('/kaggle/working')
    
    WORK_DIR = "gaussian-splatting"
    
    if not os.path.exists(WORK_DIR):
        print("Cloning Gaussian Splatting repository...")
        run_cmd([
            "git", "clone", "--recursive",
            "https://github.com/graphdeco-inria/gaussian-splatting.git",
            WORK_DIR
        ])
    else:
        print("✓ Repository already exists")
    
    os.chdir(WORK_DIR)
    
    # Install requirements
    print("Installing Gaussian Splatting requirements...")
    run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    
    # Build submodules
    print("\n📦 Building Gaussian Splatting submodules...")
    
    submodules = {
        "diff-gaussian-rasterization":
            "https://github.com/graphdeco-inria/diff-gaussian-rasterization.git",
        "simple-knn":
            "https://github.com/camenduru/simple-knn.git"
    }
    
    for name, repo in submodules.items():
        print(f"\n📦 Installing {name}...")
        path = os.path.join("submodules", name)
        if not os.path.exists(path):
            run_cmd(["git", "clone", repo, path])
        run_cmd([sys.executable, "-m", "pip", "install", path])
    
    print("✓ Gaussian Splatting setup complete!")


def train_gaussian_splatting(colmap_dir, image_dir, output_dir, iterations=2000):
    """Train Gaussian Splatting model"""
    print("\n" + "="*70)
    print("Step 6: Training Gaussian Splatting")
    print("="*70)
    
    print("\n=== Training Gaussian Splatting ===")
    
    # Reduce memory usage with smaller resolution
    cmd = [
        'python', 'train.py',
        '-s', colmap_dir,
        '--images', image_dir,
        '-m', output_dir,
        '--iterations', str(iterations),
        '--test_iterations', '1000', str(iterations),
        '--save_iterations', '1000', str(iterations),
        '--resolution', '2',  # Reduce resolution to 1/2
        '--densify_grad_threshold', '0.001',  # Higher threshold = fewer Gaussians
        '--densification_interval', '200',  # Less frequent densification
        '--opacity_reset_interval', '5000',  # Less frequent reset
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    
    result = subprocess.run(
        cmd,
        cwd='/kaggle/working/gaussian-splatting',
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    if result.returncode != 0:
        raise RuntimeError("Gaussian Splatting training failed")
    
    # Check output
    if not os.path.exists(os.path.join(output_dir, f'point_cloud/iteration_{iterations}/point_cloud.ply')):
        raise RuntimeError(f"Expected output not found at iteration {iterations}")
    
    print(f"\n✓ Gaussian Splatting training completed successfully")
    print(f"  Output: {output_dir}")
    
    return output_dir

# Main pipeline


This code defines a comprehensive computational pipeline designed to transform a collection of standard photographs into a detailed 3D reconstruction. The process begins by normalizing images and using deep learning models to select the best pairs for matching, which ensures the system efficiently understands the spatial relationships between different viewpoints. By integrating advanced frameworks like MASt3R and COLMAP, the software translates 2D data into a structured 3D point cloud and camera poses. Ultimately, these results are used to train a Gaussian Splatting model, which creates a high-fidelity, renderable digital environment from the original visual input.

In [7]:
# ============================================================================
# Main Pipeline
# ============================================================================
def main_pipeline(image_dir, output_dir, square_size=224, iterations=2000, 
                 max_images=None, max_pairs=10000, max_points=1000000):
    """
    Main pipeline for DINO matching -> MASt3R -> Gaussian Splatting
    
    Args:
        image_dir: Directory containing input images
        output_dir: Directory for output files
        square_size: Size to resize images for processing
        iterations: Number of training iterations
        max_images: Maximum number of images to process (None = all)
        max_pairs: Maximum number of image pairs for matching
        max_points: Maximum number of 3D points to extract (default: 1M)
    """
    os.makedirs(output_dir, exist_ok=True)

    setup_base_environment()
    clear_memory()
    
    setup_mast3r()
    clear_memory()
    
    setup_gaussian_splatting()
    clear_memory()
    
    # Step 1: Normalize images to biplet-square format
    print("\n" + "="*70)
    print("Step 1: Biplet-Square Normalization")
    print("="*70)
    
    processed_image_dir = os.path.join(output_dir, "processed_images")
    
    # Get original images first
    original_image_paths = sorted([
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    
    # Limit original images if specified
    if max_images and len(original_image_paths) > max_images:
        print(f"\n⚠️  Limiting to {max_images} original images")
        original_image_paths = original_image_paths[:max_images]
    
    print(f"Processing {len(original_image_paths)} original images → ~{len(original_image_paths)*2} after biplet-square")
    
    # Only process the selected images
    temp_dir = os.path.join(output_dir, "temp_originals")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Copy selected images to temp directory
    for img_path in original_image_paths:
        import shutil
        shutil.copy(img_path, temp_dir)
    
    # Process the temp directory
    normalize_image_sizes_biplet(
        input_dir=temp_dir,
        output_dir=processed_image_dir,
        size=square_size
    )
    
    # Clean up temp directory
    shutil.rmtree(temp_dir)
    
    # Get processed image paths
    image_paths = sorted([
        os.path.join(processed_image_dir, f)
        for f in os.listdir(processed_image_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    
    print(f"\n📸 Processing {len(image_paths)} images (after biplet-square)")
    print(f"⚠️  Will use maximum {max_pairs} pairs to save memory")
    
    # Step 2: DINO-based pair selection
    print("\n" + "="*70)
    print("Step 2: DINO Pair Selection")
    print("="*70)
    
    pairs = get_image_pairs_dino(image_paths, max_pairs=max_pairs)
    clear_memory()
    
    print(f"✓ Using {len(pairs)} pairs for reconstruction")
    
    # Step 3: MASt3R reconstruction
    print("\n" + "="*70)
    print("Step 3: MASt3R Reconstruction")
    print("="*70)
    
    device = Config.DEVICE
    model = load_mast3r_model(device)
    
    scene, mast3r_images = run_mast3r_pairs(
        model, image_paths, pairs, device,
        max_pairs=None  # Already limited in get_image_pairs_dino
    )
    
    # Clear model from memory
    del model
    clear_memory()
    
    # Step 4: Extract COLMAP-compatible data
    print("\n" + "="*70)
    print("Step 4: Converting to COLMAP Format")
    print("="*70)
    
    # Extract COLMAP-compatible data with point limit
    pts3d, colors, cameras, poses = extract_colmap_data(
        scene, image_paths, max_points=max_points  
    )

    # Clear scene from memory
    del scene, mast3r_images
    clear_memory()
    
    # Step 5: Save COLMAP reconstruction
    colmap_dir = os.path.join(output_dir, 'colmap')
    sparse_dir = save_colmap_reconstruction(
        pts3d, colors, cameras, poses, image_paths, colmap_dir
    )
    
    # Clear reconstruction data
    del pts3d, colors, cameras, poses
    clear_memory()
    
    # Step 6: Train Gaussian Splatting
    print("\n" + "="*70)
    print("Step 6: Training Gaussian Splatting")
    print("="*70)
    
    gs_output = train_gaussian_splatting(
        colmap_dir=colmap_dir,
        image_dir=processed_image_dir,
        output_dir=output_dir,
        iterations=iterations
    )
    
    print("\n" + "="*70)
    print("✅ Full Pipeline Successfully Completed!")
    print("="*70)
    print(f"\nGaussian Splatting model saved at: {gs_output}")
    
    return gs_output


if __name__ == "__main__":
    IMAGE_DIR = "/kaggle/input/two-dogs/bike15"
    OUTPUT_DIR = "/kaggle/working/output"
    
    gs_output = main_pipeline(
        image_dir=IMAGE_DIR,
        output_dir=OUTPUT_DIR,
        square_size=1024,  
        iterations=1000,   
        max_images=30,
        max_pairs=1000,     
        max_points=1000000        
    )

    print(f"\n{'='*70}")
    print("Pipeline completed successfully!")
    print(f"{'='*70}")
    print(f"Gaussian Splatting output: {gs_output}")


=== Setting up Base Environment ===

📦 Fixing NumPy...
Running: /usr/bin/python3 -m pip uninstall -y numpy
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Running: /usr/bin/python3 -m pip install numpy==1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2


📦 Installing PyTorch...
Running: /usr/bin/python3 -m pip install torch torchvision torchaudio

📦 Installing core utilities...
Running: /usr/bin/python3 -m pip install opencv-python pillow imageio imageio-ffmpeg plyfile tqdm tensorboard scipy psutil
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 88.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatibl


📦 Installing transformers...
Running: /usr/bin/python3 -m pip install transformers==4.40.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1


...

Running MASt3R inference on 290 pairs...
>> Inference with model on 290 image pairs


  0%|          | 0/290 [00:00<?, ?it/s]/kaggle/working/mast3r/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/kaggle/working/mast3r/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/kaggle/working/mast3r/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 290/290 [01:07<00:00,  4.29it/s]


✓ MASt3R inference complete
After inference:
GPU Memory - Allocated: 2.58GB, Reserved: 2.69GB
CPU Memory Usage: 30.8%
Running global alignment...
Computing global alignment...
 init edge (2*,4*) score=np.float64(48.25545120239258)
 init edge (2,22*) score=np.float64(31.884056091308594)
 init edge (0*,22) score=np.float64(31.374319076538086)
 init edge (22,28*) score=np.float64(30.269298553466797)
 init edge (3*,4) score=np.float64(16.28212547302246)
 init edge (8*,22) score=np.float64(43.51755142211914)
 init edge (5*,22) score=np.float64(36.874473571777344)
 init edge (14*,22) score=np.float64(33.16047286987305)
 init edge (14,16*) score=np.float64(32.07289505004883)
 init edge (12*,16) score=np.float64(31.95780372619629)
 init edge (1*,5) score=np.float64(29.714458465576172)
 init edge (14,17*) score=np.float64(23.525936126708984)
 init edge (13*,17) score=np.float64(20.83549690246582)
 init edge (15*,17) score=np.float64(17.212093353271484)
 init edge (5,23*) score=np.float64(41.186

  0%|          | 0/150 [00:00<?, ?it/s]/kaggle/working/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:366: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  return float(loss), lr
100%|██████████| 150/150 [00:20<00:00,  7.25it/s, lr=2.09647e-06 loss=0.0431117]


✓ Global alignment complete (final loss: 0.043112)
Final memory state:
GPU Memory - Allocated: 3.49GB, Reserved: 5.30GB
CPU Memory Usage: 21.6%

Step 4: Converting to COLMAP Format

=== Extracting COLMAP-compatible data ===
pts_all type: <class 'list'>
pts_all is a list with 30 elements
First element type: <class 'torch.Tensor'>
First element shape: torch.Size([224, 224, 3])
pts_all shape after conversion: torch.Size([30, 224, 224, 3])
Found batched point cloud: torch.Size([30, 224, 224, 3])
✓ Extracted 1505280 3D points from 30 images

⚠ Downsampling from 1505280 to 1000000 points to reduce memory usage...
✓ Downsampled to 1000000 points
Extracting camera parameters...
Retrieved camera-to-world poses: shape (30, 4, 4)
Converted to world-to-camera poses for COLMAP
Focals shape: (30, 1)
Principal points shape: (30, 2)

Example camera 0:
  Image size: 1024x1024
  MASt3R focal: 260.02, pp: (112.00, 112.00)
  Scaled fx=1188.66, fy=1188.66, cx=512.00, cy=512.00
  Pose (first row): [ 0.83565

## **3D Gaussian Splat Viewer**

https://splat-three.vercel.app/?url=bike_mast3r_ps1.splat

https://splat-three.vercel.app/?url=bike_mast3r_ps1.splat#[0.61,-0.35,0.71,0,0.26,0.93,0.24,0,-0.76,0.04,0.65,0,0.53,0.08,0.9,1]
